In [ ]:
# ────────────────────────────────────────────────
# 1. Standard library & third-party imports
# ────────────────────────────────────────────────
import sys
import os
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import rasterio
from rasterio.plot import show
from rasterio.warp import calculate_default_transform, reproject, Resampling
import pyproj
import plotly.graph_objects as go

# ANUGA & related
import anuga
from anuga import Domain_plotter

# Jupyter / animation support
from IPython.display import HTML
import matplotlib.animation as animation
from matplotlib import rc

%matplotlib inline

# ────────────────────────────────────────────────
# 2. Add project paths (robust for notebooks)
# ────────────────────────────────────────────────
notebook_dir = Path.cwd().resolve()
project_root = notebook_dir.parent.parent  # adjust levels as needed

sys.path.insert(0, str(notebook_dir))
sys.path.insert(0, str(notebook_dir.parent))
sys.path.insert(0, str(project_root))

# ────────────────────────────────────────────────
# 3. Custom helpers / local modules
# ────────────────────────────────────────────────
from lib.geo_plots import * 

In [ ]:
topography_file = 'DEM_data/chowilla_dem_single.asc'
polyline_csv='DEM_data/chowilla_finer_area.csv'
points_csv='DEM_data/chowilla_BCs.csv'



plot_dem_ascii(
    dem_path=topography_file,
    polyline_csv=polyline_csv,
    points_csv=points_csv,
    title="Flood Plain DEM for Chowilla River"
)


In [ ]:
topography_file = 'DEM_data/chowilla_dem_single.asc'
polyline_csv='DEM_data/chowilla_finer_area.csv'
points_csv='DEM_data/chowilla_BCs.csv'


plot_dem_plotly(
    dem_path=topography_file,
    polygon_csv=polyline_csv,
    points_csv=points_csv,
    title="Flood Plain DEM for Chowilla River"
)


In [ ]:
finer_zone = anuga.read_polygon(polyline_csv)

base_resolution = 30_000.0

# # Create mesh and save to file
# boundary_tags = {}
mesh_file = 'DEM_data/Chowilla_River_mesh.msh'
boundary_tags = {}

boundary_tags = {
    'wall': list(range(0, 19)) + list(range(23, 32)),
    'outflow': [19, 20, 21, 22]
}



# boundary_tags.update({'exit': [31]})

domain = anuga.create_domain_from_regions(
    bounding_polygon=finer_zone,
    boundary_tags=boundary_tags,
    maximum_triangle_area=base_resolution,
    mesh_filename=mesh_file,
    verbose=False
)

print(f"Mesh created:\n{domain.number_of_triangles} triangles\n{domain.get_number_of_nodes()} nodes")
print(f"Saved to: {mesh_file}")
domain.set_name('Chowilla_River_Steady_state')
domain.set_datadir('DEM_data')


In [ ]:
dplotter = anuga.Domain_plotter(domain)
plot_mesh(
    domain=domain,
    dplotter=dplotter,                      # your ANUGA domain object
    points_csv=points_csv,
    polyline_csv=None,
    radius=100,
    figsize=(7, 5),
    title="Chowilla River mesh"
)

In [ ]:
# Low stage value — make it negative enough so water can flow out freely
low_stage = -10.0

# Outflow boundary: low stage + zero momentum
# outflow_BC = anuga.Dirichlet_boundary([low_stage, 0.0, 0.0])

wall_BC = anuga.Reflective_boundary(domain)
outflow_BC = anuga.Transmissive_boundary(domain)



# Apply outflow to ALL exterior boundaries
domain.set_boundary({
    'wall': wall_BC,          # ← change 'bc' to your actual boundary tag name
    'outflow': outflow_BC,
    # or 'boundary': outflow_BC,
})

print("All boundaries set to outflow (Dirichlet with low stage).")

In [ ]:
import anuga
import pandas as pd
import numpy as np


# ────────────────────────────────────────────────
# CONFIG
# ────────────────────────────────────────────────
Q_CONST = 200.0          # m3/s per inlet
N_INLETS = 3             # ONLY first 3 points
SAFETY = 1.2             # 20% margin on radius


# ────────────────────────────────────────────────
# LOAD FIRST 3 POINTS ONLY
# ────────────────────────────────────────────────
points_df = pd.read_csv(
    points_csv,
    header=None,
    names=["x", "y"]
).iloc[:N_INLETS]

print(f"Using {len(points_df)} inlet point(s) (first {N_INLETS} only)")


# ────────────────────────────────────────────────
# PRECOMPUTE CENTROIDS ONCE
# ────────────────────────────────────────────────
centroids = domain.get_centroid_coordinates()


# ────────────────────────────────────────────────
# ASSIGN INLETS (AUTO RADIUS)
# ────────────────────────────────────────────────
inlets = []

for i, row in points_df.iterrows():
    center = (float(row["x"]), float(row["y"]))
    c = np.array(center)

    # Distance to nearest centroid
    d = np.sqrt(((centroids - c) ** 2).sum(axis=1))
    radius_used = d.min() * SAFETY

    region = anuga.Region(
        domain,
        center=center,
        radius=radius_used
    )

    inlet = anuga.Inlet_operator(
        domain,
        region,
        Q=Q_CONST
    )

    inlets.append(inlet)

    print(
        f"Inlet {i+1}: "
        f"Center=({center[0]:.1f}, {center[1]:.1f}), "
        f"Radius_used={radius_used:.2f} m, "
        f"Q={Q_CONST} m³/s"
    )


print(
    f"\nSuccess: {len(inlets)} inlets created "
    f"(first {N_INLETS} CSV points, Q = {Q_CONST} m³/s each)"
)


In [ ]:
domain.set_quantity('elevation', filename=topography_file, location='centroids') # Use function for elevation
domain.set_quantity('friction', 0.06, location='centroids')                        # Constant friction 
domain.set_quantity('stage', expression='elevation', location='centroids')         # Dry Bed 

In [ ]:
# 1. Define time steps
yield_step = 1 * 24 * 3600  # Yield every 30 minutes (1800s)
save_step  = 1 * 24 * 3600  # Save image every 60 minutes (3600s)
final_time = 40 * 24 * 3600   # Total 24 hours

print("Starting Simulation: Yielding every 0.5hr, Saving every 1hr...")

for t in domain.evolve(yieldstep=yield_step, finaltime=final_time):
    
    # Print statistics every 30 minutes (on every yield)
    domain.print_timestepping_statistics()
    
    # 2. Logic to save only every 1 hour
    # We check if the current time t is a multiple of 3600
    if t % save_step == 0:
        print(f"--- Saving Depth Frame at Time: {t/3600:.1f} hours ---")
        dplotter.save_depth_frame(vmin=0.05, vmax=5)
        # Use formatted string to show hours and current Q
    print(f"Time: {t/3600:>4.1f}h")

print("Simulation Complete.")

In [ ]:

rc('animation', html='jshtml')
anim = dplotter.make_depth_animation()
anim